# TP2 Notebook 01 : MongoDB et simulation des paiements

Sekou Camara, B2 Sup de Vinci Rennes

On récupère les données de l'API DummyJSON et le fichier des commandes, on les range dans MongoDB (base `commerce`), puis on simule un paiement par panier.

## 0. Connexion à MongoDB

On se connecte à MongoDB qui tourne dans Docker sur le port 27017. Si on voit `{'ok': 1.0}`, la base répond.

In [1]:
import requests
import numpy as np
import pandas as pd
from datetime import datetime
from pymongo import MongoClient, ASCENDING

MONGO_URI = "mongodb://localhost:27017"
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
print(client.admin.command("ping"))
print("Version MongoDB :", client.server_info()["version"])
db = client["commerce"]

{'ok': 1.0}
Version MongoDB : 7.0.48


## 1. Charger les 8 ressources de l'API dans MongoDB

Pour chaque ressource, on appelle l'API avec `limit=0` pour tout récupérer d'un coup. On vide la collection avant d'insérer, comme ça on peut relancer sans créer de doublons. Ensuite on vérifie que le nombre reçu est le même que le `total` annoncé par l'API.

In [2]:
BASE_URL = "https://dummyjson.com"
RESSOURCES = ["products", "carts", "users", "posts", "comments", "recipes", "quotes", "todos"]


def ingerer(ressource):
    reponse = requests.get(f"{BASE_URL}/{ressource}", params={"limit": 0}, timeout=30)
    reponse.raise_for_status()
    contenu = reponse.json()
    documents = contenu[ressource]
    collection = db[f"raw_{ressource}"]
    collection.drop()
    if documents:
        collection.insert_many(documents)
    return {
        "ressource": ressource,
        "collection": f"raw_{ressource}",
        "total_api": contenu.get("total", len(documents)),
        "recus": len(documents),
        "en_base": collection.count_documents({}),
    }


bilan_api = pd.DataFrame([ingerer(r) for r in RESSOURCES])
bilan_api["controle_ok"] = (bilan_api["total_api"] == bilan_api["recus"]) & (bilan_api["recus"] == bilan_api["en_base"])
bilan_api

,ressource,collection,total_api,recus,en_base,controle_ok
0,products,raw_products,194,194,194,True
1,carts,raw_carts,208,208,208,True
2,users,raw_users,208,208,208,True
3,posts,raw_posts,251,251,251,True
4,comments,raw_comments,340,340,340,True
5,recipes,raw_recipes,50,50,50,True
6,quotes,raw_quotes,1454,1454,1454,True
7,todos,raw_todos,254,254,254,True


In [3]:
for nom in sorted(db.list_collection_names()):
    print(f"{nom:<16} {db[nom].count_documents({}):>6} documents")

ind_ca_par_categorie     24 documents
ind_evolution_mensuelle     10 documents
ind_repartition_statuts      3 documents
ind_taux_refus        1 documents
ind_top10_clients     10 documents
raw_carts           208 documents
raw_commandes       800 documents
raw_comments        340 documents
raw_payments        208 documents
raw_posts           251 documents
raw_products        194 documents
raw_quotes         1454 documents
raw_recipes          50 documents
raw_todos           254 documents
raw_users           208 documents


## 2. Importer le fichier des commandes

On s'attend à 800 lignes, 16 colonnes et 208 paniers.

In [4]:
df = pd.read_csv("commandes_ecommerce.csv")
df.columns = [c.strip() for c in df.columns]
print("Dimensions :", df.shape)
print(list(df.columns))
df.head()

Dimensions : (800, 16)
['order_id', 'cart_id', 'user_id', 'username', 'product_id', 'product_title', 'category', 'brand', 'quantity', 'unit_price', 'line_total', 'discount_percentage', 'discounted_line_total', 'cart_total', 'cart_discounted_total', 'order_date']


,order_id,cart_id,user_id,username,product_id,product_title,category,brand,quantity,unit_price,line_total,discount_percentage,discounted_line_total,cart_total,cart_discounted_total,order_date
0,CART-00001,1,1,emilys,162,Blue Frock,tops,NaN,4,29.99,119.96,12.13,105.41,13037.88,11510.81,2025-03-02
1,CART-00001,1,1,emilys,113,Generic Motorcycle,motorcycle,Generic Motors,3,3999.99,11999.97,12.10,10547.97,13037.88,11510.81,2025-06-13
2,CART-00001,1,1,emilys,122,iPhone 6,smartphones,Apple,3,299.99,899.97,6.69,839.76,13037.88,11510.81,2025-09-15
3,CART-00001,1,1,emilys,138,Baseball Ball,sports-accessories,NaN,2,8.99,17.98,1.71,17.67,13037.88,11510.81,2025-09-20
4,CART-00002,2,2,michaelw,86,Man Short Sleeve Shirt,mens-shirts,Casual Comfort,5,19.99,99.95,6.83,93.12,139.93,125.70,2025-11-28


On transforme la colonne `order_date` en vraie date, sinon pandas la garde en texte.

In [5]:
df["order_date"] = pd.to_datetime(df["order_date"])
df.dtypes

order_id                            str
cart_id                           int64
user_id                           int64
username                            str
product_id                        int64
product_title                       str
category                            str
brand                               str
quantity                          int64
unit_price                      float64
line_total                      float64
discount_percentage             float64
discounted_line_total           float64
cart_total                      float64
cart_discounted_total           float64
order_date               datetime64[us]
dtype: object

### Vérifications avant l'import

In [6]:
controles_csv = {
    "lignes": len(df),
    "colonnes": df.shape[1],
    "paniers": df["cart_id"].nunique(),
    "clients": df["user_id"].nunique(),
    "lignes en double": int(df.duplicated().sum()),
    "valeurs manquantes": int(df.isna().sum().sum()),
    "montants negatifs": int((df["discounted_line_total"] < 0).sum()),
}
controles_csv

{'lignes': 800,
 'colonnes': 16,
 'paniers': 208,
 'clients': 208,
 'lignes en double': 0,
 'valeurs manquantes': 377,
 'montants negatifs': 0}

In [7]:
df.isna().sum()[df.isna().sum() > 0]

brand    377
dtype: int64

### Insertion dans `raw_commandes`

MongoDB n'accepte pas les `NaN` de pandas, donc on les remplace par `None` avant d'insérer. Après l'insertion, on recompte pour être sûr que les 800 lignes sont bien arrivées.

In [8]:
documents = df.astype(object).where(df.notna(), None).to_dict("records")

commandes = db["raw_commandes"]
commandes.drop()
commandes.insert_many(documents)

print("Documents en base :", commandes.count_documents({}))
print("Paniers en base :", len(commandes.distinct("cart_id")))

Documents en base : 800
Paniers en base : 208


In [9]:
somme_csv = round(df["discounted_line_total"].sum(), 2)
somme_mongo = list(commandes.aggregate([{"$group": {"_id": None, "total": {"$sum": "$discounted_line_total"}}}]))[0]["total"]
print("Somme des montants dans le CSV :", somme_csv)
print("Somme des montants dans MongoDB :", round(somme_mongo, 2))

Somme des montants dans le CSV : 3456709.58
Somme des montants dans MongoDB : 3456709.58


## 3. Simuler les paiements

Comment on fait :

1. On regroupe les 800 lignes par `cart_id`. On obtient une ligne par panier, donc 208 paniers et un seul paiement par panier.
2. Pour chaque panier on garde le client (`user_id`) et le montant, qui est la somme des `discounted_line_total` du panier. On n'utilise pas `cart_total` parce qu'il est répété sur chaque ligne.
3. On tire le statut au hasard : PAID 70 %, REFUSED 20 %, PENDING 10 %. La graine 42 permet de retrouver les mêmes résultats à chaque fois.
4. La date du paiement est la date la plus ancienne des lignes du panier.
5. Dans MongoDB on met un index unique sur `cart_id` : impossible d'enregistrer deux paiements pour le même panier.

In [10]:
rng = np.random.default_rng(42)

paiements = df.groupby("cart_id").agg(
    user_id=("user_id", "first"),
    amount=("discounted_line_total", "sum"),
    nb_lignes=("product_id", "count"),
    attempted_at=("order_date", "min"),
).reset_index()

paiements["amount"] = paiements["amount"].round(2)
paiements["status"] = rng.choice(["PAID", "REFUSED", "PENDING"], size=len(paiements), p=[0.7, 0.2, 0.1])
paiements["payment_id"] = ["PAY" + str(i).zfill(5) for i in paiements["cart_id"]]
paiements["is_cashed"] = paiements["status"] == "PAID"
paiements.head()

,cart_id,user_id,amount,nb_lignes,attempted_at,status,payment_id,is_cashed
0,1,1,11510.81,4,2025-03-02,REFUSED,PAY00001,False
1,2,2,125.70,2,2025-02-22,PAID,PAY00002,True
2,3,3,1625.77,6,2025-04-25,REFUSED,PAY00003,False
3,4,4,636.81,2,2025-09-09,PAID,PAY00004,True
4,5,5,1205.80,3,2025-05-03,PAID,PAY00005,True


In [11]:
raw_payments = db["raw_payments"]
raw_payments.drop()
raw_payments.create_index("cart_id", unique=True)
raw_payments.insert_many(paiements.astype(object).to_dict("records"))
print("Paiements en base :", raw_payments.count_documents({}))

Paiements en base : 208


### Vérifications des paiements

In [12]:
doublons = list(raw_payments.aggregate([
    {"$group": {"_id": "$cart_id", "n": {"$sum": 1}}},
    {"$match": {"n": {"$gt": 1}}},
]))
total_paiements = list(raw_payments.aggregate([{"$group": {"_id": None, "total": {"$sum": "$amount"}}}]))[0]["total"]

print("Nombre de paiements :", raw_payments.count_documents({}), "pour", df["cart_id"].nunique(), "paniers")
print("Paniers payés deux fois :", len(doublons))
print("Statuts présents :", raw_payments.distinct("status"))
print("Montants négatifs ou nuls :", raw_payments.count_documents({"amount": {"$lte": 0}}))
print("Somme des paiements :", round(total_paiements, 2), "et somme des lignes :", round(df["discounted_line_total"].sum(), 2))

Nombre de paiements : 208 pour 208 paniers
Paniers payés deux fois : 0
Statuts présents : ['PAID', 'PENDING', 'REFUSED']
Montants négatifs ou nuls : 0
Somme des paiements : 3456709.58 et somme des lignes : 3456709.58


In [13]:
panier_existant = raw_payments.find_one()["cart_id"]
try:
    raw_payments.insert_one({"payment_id": "TEST", "cart_id": panier_existant, "status": "PAID"})
    print("Le doublon a été accepté")
except Exception as erreur:
    print("Doublon refusé par MongoDB :", type(erreur).__name__)

Doublon refusé par MongoDB : DuplicateKeyError


## 4. Analyser les résultats

In [14]:
repartition = pd.DataFrame(list(raw_payments.aggregate([
    {"$group": {"_id": "$status", "nb_paiements": {"$sum": 1}, "montant_total": {"$sum": "$amount"}}},
    {"$sort": {"nb_paiements": -1}},
]))).rename(columns={"_id": "status"})
repartition["pourcentage"] = (100 * repartition["nb_paiements"] / repartition["nb_paiements"].sum()).round(1)
repartition["montant_total"] = repartition["montant_total"].round(2)
repartition

,status,nb_paiements,montant_total,pourcentage
0,PAID,148,2396066.11,71.2
1,REFUSED,43,804575.17,20.7
2,PENDING,17,256068.30,8.2


In [15]:
encaisse = repartition.loc[repartition["status"] == "PAID", "montant_total"].sum()
total = repartition["montant_total"].sum()
print("Tentatives de paiement :", repartition["nb_paiements"].sum())
print("Montant total tenté :", round(total, 2), "USD")
print("Montant encaissé (PAID) :", round(encaisse, 2), "USD, soit", round(100 * encaisse / total, 1), "%")
print("Montant non encaissé :", round(total - encaisse, 2), "USD")

Tentatives de paiement : 208
Montant total tenté : 3456709.58 USD
Montant encaissé (PAID) : 2396066.11 USD, soit 69.3 %
Montant non encaissé : 1060643.47 USD


### Réponses aux questions

**Comment avoir un seul paiement par panier ?** On regroupe les lignes par `cart_id` avec `groupby`. Chaque panier devient une seule ligne, donc un seul paiement. L'index unique dans MongoDB bloque un deuxième paiement pour le même panier.

**Comment respecter 70 / 20 / 10 ?** On fait un tirage au hasard avec ces probabilités pour chaque panier. Sur 208 paniers on tombe à peu près sur ces pourcentages mais pas pile, c'est normal avec si peu de tirages.

**Quelles infos du panier on garde ?** `cart_id` pour la commande, `user_id` pour le client, et le montant calculé avec la somme des `discounted_line_total`.

**Quels contrôles ?** 208 paiements pour 208 paniers, aucun panier payé deux fois, seulement les 3 statuts prévus, pas de montant négatif, la somme des paiements égale à la somme des lignes, et un doublon refusé par MongoDB.

**Résultats :** 148 PAID, 43 REFUSED, 17 PENDING. Seuls les PAID sont de l'argent vraiment encaissé : 2 396 066,11 USD sur 3 456 709,58 USD tentés.

In [16]:
for nom in sorted(db.list_collection_names()):
    print(f"{nom:<16} {db[nom].count_documents({}):>6} documents")

ind_ca_par_categorie     24 documents
ind_evolution_mensuelle     10 documents
ind_repartition_statuts      3 documents
ind_taux_refus        1 documents
ind_top10_clients     10 documents
raw_carts           208 documents
raw_commandes       800 documents
raw_comments        340 documents
raw_payments        208 documents
raw_posts           251 documents
raw_products        194 documents
raw_quotes         1454 documents
raw_recipes          50 documents
raw_todos           254 documents
raw_users           208 documents
